# Análisis de Ruido: Puerto USB (PC) vs. Batería 18650 (Módulo J5019)
Este notebook genera datos simulados *(mockups)* en formato CSV equivalentes a 5 segundos de toma de bioseñales en las manos, y extrae métricas de calidad (SNR y Pico a Pico).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import os

# Configuraciones Globales de la Interfaz Estética (Dark Mode)
plt.style.use('dark_background')

# Parámetros de la simulación
fs = 250  # Frecuencia de muestreo (250 Hz)
t = np.arange(0, 5, 1/fs)  # 5 segundos de señal temporizada

# ---------------- 1. CONSTRUCCIÓN DE SEÑAL LIMPIA BASE ---------------- #
# Generando una bioseñal (similar a ECG o pulso) limpia en ~60BPM
clean_signal = signal.sawtooth(2 * np.pi * 1.0 * t, 0.1) * 0.4 
clean_signal += np.sin(2 * np.pi * 1.0 * t) * 0.15
clean_signal[clean_signal < 0] = clean_signal[clean_signal < 0] * 0.3 # Ajuste de forma

# ---------------- 2. SIMULACIÓN DEL ENTORNO ---------------- #

# Caso A: Conectado a la PC (Bucle de tierra -> Ruido 60Hz Dominante + Ruido ATX)
noise_60hz = np.sin(2 * np.pi * 60 * t) * 0.35      # Interferencia electromagnética red eléctrica
noise_atx = np.random.normal(0, 0.08, len(t))        # Ripple de placa madre USB
signal_pc = clean_signal + noise_60hz + noise_atx

# Caso B: Conectado a Batería con Mod G5019 (Aislamiento Galvánico = No 60Hz, Doble filtrado mata alta freq)
# El Boost converter hace ruido rápido que ya fue filtrado. Nos queda un ligero ruido térmico.
noise_thermal = np.random.normal(0, 0.012, len(t))   # Ruido térmico remanente 
signal_bat = clean_signal + noise_thermal

# ---------------- 3. CREAR Y EXPORTAR CSVS ---------------- #
df_pc = pd.DataFrame({'time_s': t, 'voltage_mV': signal_pc})
df_bat = pd.DataFrame({'time_s': t, 'voltage_mV': signal_bat})

df_pc.to_csv('lecturas_pc_usb_5s.csv', index=False)
df_bat.to_csv('lecturas_bat_j5019_5s.csv', index=False)

print("Los archivos 'lecturas_pc_usb_5s.csv' y 'lecturas_bat_j5019_5s.csv' fueron creados exitosamente.")

### Comparación Objetiva de las Señales (Carga de CSV)

In [ ]:
# Cargar los archivos generados como si fuese un análisis en diferido
data_pc = pd.read_csv('lecturas_pc_usb_5s.csv')
data_bat = pd.read_csv('lecturas_bat_j5019_5s.csv')

def extract_objective_metrics(signal_ideal, signal_real):
    """Calcula SNR, Ruido RMS y Ruido Pico a Pico para ser objetivos"""
    noise = signal_real - signal_ideal
    rms_signal = np.sqrt(np.mean(signal_ideal**2))
    rms_noise = np.sqrt(np.mean(noise**2))
    # SNR (Relación Señal-Ruido) en Decibelios (dB)
    snr_db = 20 * np.log10(rms_signal / rms_noise) if rms_noise != 0 else float('inf')
    # Ruido Pico a Pico
    peak_to_peak = np.max(noise) - np.min(noise)
    return rms_noise, snr_db, peak_to_peak

rms_pc, snr_pc, vp_pc = extract_objective_metrics(clean_signal, data_pc['voltage_mV'].values)
rms_bat, snr_bat, vp_bat = extract_objective_metrics(clean_signal, data_bat['voltage_mV'].values)

print("================ REPORTE DE CALIDAD DE SEÑAL (5S) =================")
print("\n🔌 FUENTE 1: PUERTO USB DE PC")
print(f"   - Ruido Promedio (RMS):      {rms_pc:.4f} mV")
print(f"   - Ruido Max/Min (Pico a Pico): {vp_pc:.4f} mV")
print(f"   - Relación Señal a Ruido:    {snr_pc:.2f} dB (Menor es peor calidad)")

print("\n🔋 FUENTE 2: BATERÍA 18650 + MÓDULO J5019")
print(f"   - Ruido Promedio (RMS):      {rms_bat:.4f} mV")
print(f"   - Ruido Max/Min (Pico a Pico): {vp_bat:.4f} mV")
print(f"   - Relación Señal a Ruido:    {snr_bat:.2f} dB (Mayor es mejor calidad)")
print("\nEl módulo J5019 + Batería supera críticamente al puerto PC debido al aislamiento del ruido de modo común de 60Hz.")

### Gráficas de los Datos de Sensores

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True)

# Señal PC
ax[0].plot(data_pc['time_s'], data_pc['voltage_mV'], color='#ff4c4c', alpha=0.9, linewidth=1.5)
ax[0].set_title('Toma de Manos (USB de PC) - Note la interferencia masiva 60Hz', color='white', pad=10)
ax[0].set_ylabel('Amplitud (mV)')
ax[0].grid(True, color='#333333', linestyle=':')

# Señal Batería 
ax[1].plot(data_bat['time_s'], data_bat['voltage_mV'], color='#00ff9d', alpha=0.9, linewidth=1.5)
ax[1].set_title('Toma de Manos (Bat 18650 + J5019) - Señal aislada y limpia', color='white', pad=10)
ax[1].set_xlabel('Tiempo (Segundos)')
ax[1].set_ylabel('Amplitud (mV)')
ax[1].grid(True, color='#333333', linestyle=':')

plt.tight_layout()
plt.show()